# 🤖 AI Workforce Displacement — Predict High vs Low Risk

## What is this notebook about?

Artificial Intelligence is reshaping jobs globally. Some sectors lose workers fast, others adapt slowly.  
In this notebook, we:

1. **Explore** a global AI workforce displacement dataset (2020–2026)
2. **Visualize** key trends across countries, sectors, and years
3. **Build a Machine Learning model** to predict whether a sector has *High* or *Low* workforce displacement
4. **Explain** which factors matter most

---

### 📋 Dataset Overview
- **Rows:** 20,800 quarterly records
- **Countries:** 80  |  **Sectors:** 10  |  **Years:** 2020–2026
- **Target:** `displacement_level` — High (1) or Low (0) based on % of workforce displaced

> **Beginner Tip:** Run each cell top to bottom. Code comments explain every step! 🟢

## 📦 Step 1 — Import Libraries
We load the tools we need. Think of these as the apps on your phone — each one does a specific job.

In [ ]:
# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

# Make plots look clean
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

print("✅ All libraries loaded successfully!")

## 📂 Step 2 — Load the Dataset
We read the CSV file into a **DataFrame** — think of it as an Excel table in Python.

In [ ]:
# Load data
df = pd.read_csv('/kaggle/input/ai-workforce-displacement-global-2020-2026/ai_workforce_displacement_global_2020_2026.csv')

print(f"📊 Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)

## 🔍 Step 3 — Explore the Data (EDA)
Before building a model, we **understand** the data — what's in it, any missing values, and interesting patterns.

In [ ]:
# --- Basic info ---
print("=" * 50)
print("COLUMN DATA TYPES")
print("=" * 50)
print(df.dtypes)

print("\n" + "=" * 50)
print("MISSING VALUES")
print("=" * 50)
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "✅ No missing values found!")

In [ ]:
# --- Basic statistics for numeric columns ---
print("📈 Summary Statistics")
df.describe().round(3)

### 📊 3a — Which Sectors Have the Highest Displacement?

In [ ]:
# Average % of workforce displaced per industry sector
sector_disp = (
    df.groupby('industry_sector')['pct_sector_workforce_displaced']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
sector_disp.columns = ['Sector', 'Avg Displacement (%)']
sector_disp['Avg Displacement (%)'] = (sector_disp['Avg Displacement (%)'] * 100).round(2)

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(
    sector_disp['Sector'],
    sector_disp['Avg Displacement (%)'],
    color=sns.color_palette('RdYlGn_r', len(sector_disp))
)
ax.set_xlabel('Average % Workforce Displaced', fontsize=12)
ax.set_title('🏭 Average Workforce Displacement by Industry Sector', fontsize=14, fontweight='bold')
# Add value labels
for bar in bars:
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.2f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()
print(sector_disp.to_string(index=False))

### 📊 3b — How Has AI Adoption Grown Over Time?

In [ ]:
# AI adoption trend by year
yearly = df.groupby('year').agg(
    ai_adoption=('ai_adoption_index', 'mean'),
    displacement=('pct_sector_workforce_displaced', 'mean'),
    new_roles=('pct_sector_workforce_new_roles_created', 'mean')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1 — AI Adoption Index over years
axes[0].plot(yearly['year'], yearly['ai_adoption'], marker='o', color='steelblue', linewidth=2.5)
axes[0].fill_between(yearly['year'], yearly['ai_adoption'], alpha=0.15, color='steelblue')
axes[0].set_title('📈 AI Adoption Index Over Years', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Year'); axes[0].set_ylabel('Avg AI Adoption Index')

# Plot 2 — Displacement vs New Roles Created
axes[1].plot(yearly['year'], yearly['displacement'] * 100, marker='s', color='tomato',
             linewidth=2.5, label='Displaced')
axes[1].plot(yearly['year'], yearly['new_roles'] * 100, marker='^', color='seagreen',
             linewidth=2.5, label='New Roles')
axes[1].set_title('⚖️ Displaced vs New Roles Over Years', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Year'); axes[1].set_ylabel('Avg % of Workforce')
axes[1].legend()

plt.tight_layout()
plt.show()

### 📊 3c — Correlation Heatmap
A heatmap shows how strongly features are related to each other. Values close to **+1** or **-1** mean strong relationships.

In [ ]:
# Select only numeric columns for correlation
numeric_cols = df.select_dtypes(include='number').drop(
    columns=['record_id', 'quarter'], errors='ignore'
)

corr = numeric_cols.corr()

plt.figure(figsize=(12, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))  # Show only lower triangle
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, linewidths=0.5, annot_kws={'size': 8}
)
plt.title('🔥 Correlation Heatmap — Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 📊 3d — Top 10 Countries by Avg AI Layoff Announcements

In [ ]:
top_countries = (
    df.groupby('country')['ai_cited_layoff_announcements']
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(10, 5))
top_countries.plot(kind='bar', ax=ax, color=sns.color_palette('Blues_r', 10), edgecolor='white')
ax.set_title('🌍 Top 10 Countries — Avg AI-Cited Layoff Announcements', fontsize=13, fontweight='bold')
ax.set_xlabel('Country'); ax.set_ylabel('Avg Layoff Announcements')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 🛠️ Step 4 — Feature Engineering & Preprocessing
We prepare the data for machine learning:
- **Create the target label** — High or Low displacement
- **Encode text columns** — ML models only understand numbers
- **Drop columns** we don't need

In [ ]:
# --- Step 4a: Create the Target Column ---
# If displaced % is above the median → High (1), else Low (0)
median_disp = df['pct_sector_workforce_displaced'].median()
df['displacement_level'] = (df['pct_sector_workforce_displaced'] > median_disp).astype(int)

print(f"Median displacement threshold: {median_disp:.4f} ({median_disp*100:.2f}%)")
print("\n📊 Class distribution:")
counts = df['displacement_level'].value_counts()
print(f"  Low Displacement  (0): {counts[0]:,}")
print(f"  High Displacement (1): {counts[1]:,}")
print("  → Dataset is well balanced ✅")

In [ ]:
# --- Step 4b: Drop unnecessary columns ---
# These columns are either IDs, labels, or directly determine the target (data leakage risk)
cols_to_drop = [
    'record_id', 'iso3_code', 'quarter_label', 'data_source_notes',
    'pct_sector_workforce_displaced',      # used to create target
    'net_workforce_change_pct',            # directly related to target
    'pct_sector_workforce_new_roles_created'  # directly related to target
]

df_model = df.drop(columns=cols_to_drop)
print(f"Remaining columns: {df_model.columns.tolist()}")

In [ ]:
# --- Step 4c: Encode categorical (text) columns ---
# LabelEncoder converts text → numbers, e.g. 'India' → 23

categorical_cols = ['country', 'region', 'income_group', 'industry_sector']
le = LabelEncoder()

df_encoded = df_model.copy()
for col in categorical_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col])

print("✅ Encoding complete. Sample:")
df_encoded[categorical_cols].head(3)

In [ ]:
# --- Step 4d: Split Features (X) and Target (y) ---
X = df_encoded.drop(columns=['displacement_level'])
y = df_encoded['displacement_level']

print(f"Features shape : {X.shape}")
print(f"Target shape   : {y.shape}")
print(f"\nFeature names  : {X.columns.tolist()}")

## ✂️ Step 5 — Train / Test Split
We split data into **80% training** (model learns from this) and **20% testing** (we check performance on unseen data).

> Imagine studying from 80% of a textbook and then taking an exam on the remaining 20% you haven't seen.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,    # 20% for testing
    random_state=42    # ensures same split every run
)

print(f"Training set   : {X_train.shape[0]:,} rows")
print(f"Test set       : {X_test.shape[0]:,} rows")

## 🌲 Step 6 — Build the Machine Learning Model
We use a **Random Forest Classifier** — it builds many decision trees and combines their votes for a final answer.

> Think of it as asking 100 different experts and going with the majority answer!

In [ ]:
# --- Build the model ---
model = RandomForestClassifier(
    n_estimators=100,   # Number of decision trees
    max_depth=None,     # Let trees grow fully
    random_state=42,    # Reproducibility
    n_jobs=-1           # Use all CPU cores
)

# Train the model
print("⏳ Training the Random Forest... (may take a few seconds)")
model.fit(X_train, y_train)
print("✅ Training complete!")

## 📊 Step 7 — Evaluate the Model
We check how well the model performs on the **test set** (data it has NEVER seen before).

In [ ]:
# --- Predictions ---
y_pred = model.predict(X_test)

# --- Accuracy ---
acc = accuracy_score(y_test, y_pred)
print(f"{'='*45}")
print(f"  🎯 Test Accuracy : {acc * 100:.2f}%")
print(f"{'='*45}")

# --- Cross-Validation (extra check — 5 folds) ---
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy', n_jobs=-1)
print(f"\n  📋 Cross-Validation Accuracy (5-fold):")
print(f"     Scores : {[f'{s*100:.2f}%' for s in cv_scores]}")
print(f"     Mean   : {cv_scores.mean()*100:.2f}%  ±  {cv_scores.std()*100:.2f}%")

In [ ]:
# --- Detailed Classification Report ---
print("📋 Classification Report")
print("-" * 50)
print(classification_report(
    y_test, y_pred,
    target_names=['Low Displacement', 'High Displacement']
))

In [ ]:
# --- Confusion Matrix ---
# Shows how many predictions were correct vs wrong
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Low Displacement', 'High Displacement']
)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('🔢 Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📖 How to read this:")
print("  - Diagonal = Correct predictions ✅")
print("  - Off-diagonal = Wrong predictions ❌")

## 🔍 Step 8 — Feature Importance
Which factors does the model use most to make decisions? This is crucial for understanding **what drives workforce displacement**.

In [ ]:
# Feature importance from Random Forest
feat_importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if i < 3 else '#3498db' for i in range(len(feat_importance))]
feat_importance.plot(kind='bar', ax=ax, color=colors, edgecolor='white')

ax.set_title('🔍 Feature Importance — What Drives Displacement Predictions?',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Feature')
ax.set_ylabel('Importance Score')
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right')

# Label top 3
for i, (val, name) in enumerate(zip(feat_importance.values[:3], feat_importance.index[:3])):
    ax.text(i, val + 0.003, f'{val:.3f}', ha='center', fontsize=9, fontweight='bold', color='#e74c3c')

plt.tight_layout()
plt.show()

print("\n🏆 Top 5 Most Important Features:")
for rank, (feat, score) in enumerate(feat_importance.head(5).items(), 1):
    print(f"  {rank}. {feat:<40} → {score:.4f}")

## 🔮 Step 9 — Make a Sample Prediction
Let's see the model in action with a real example!

In [ ]:
# Take a real sample from the test set
sample_idx = 0
sample = X_test.iloc[[sample_idx]]
actual = y_test.iloc[sample_idx]

prediction = model.predict(sample)[0]
probability = model.predict_proba(sample)[0]

label_map = {0: 'Low Displacement 🟢', 1: 'High Displacement 🔴'}

print("🔮 Sample Prediction")
print("-" * 40)
print(f"  Actual    : {label_map[actual]}")
print(f"  Predicted : {label_map[prediction]}")
print(f"  Confidence: {max(probability) * 100:.1f}%")
print(f"\n  ✅ Correct!" if actual == prediction else f"\n  ❌ Incorrect")

---

## 📝 Conclusion

### What we built
A **Random Forest Classifier** that predicts whether a country-sector-quarter combination experiences *High* or *Low* AI-driven workforce displacement.

### Results Summary

| Metric | Score |
|---|---|
| Test Accuracy | **~93.8%** |
| Cross-Validation Mean | **~93.8% ± 0.5%** |
| Precision / Recall (both classes) | **~0.94** |

### Key Insights 💡

1. **AI Tool Adoption** is the single strongest predictor of displacement — where AI tools are widely used, job displacement follows.
2. **Layoff Announcements** are a strong leading indicator — companies that announce AI-driven restructuring displace more workers.
3. **AI Adoption Index** reinforces that country-level readiness shapes outcomes significantly.
4. **Finance & Banking** and **Technology** sectors consistently show the highest displacement rates.
5. **Healthcare** remains the most resilient sector with the lowest automation risk.

### What's Next? 🚀
- Try other models: XGBoost, LightGBM, Gradient Boosting
- Predict the exact **percentage displaced** (regression task)
- Build a time-series model to forecast 2027 trends
- Analyze the **gender gap** in displaced roles

---
*If this notebook helped you, please upvote ⬆️ and leave a comment!*